# 02 — Full training on Colab (Loan Approval Predictor)

Run the real pipeline on Colab instead of a laptop. Full guide in `COLAB.md`.
Assumes this notebook lives in the repo (`notebooks/`) next to `src/` and `requirements.txt`.
Two ways to get the repo onto Colab: **A)** push to GitHub and clone (recommended),
or **B)** upload the repo as a zip. Data: Kaggle API (`kaggle.json`) or manual CSV upload.

In [ ]:
# 0) Environment sanity: Python should be 3.11.x for the pinned requirements.
import sys
print(sys.version)
!free -h | head -3
# If Python is not 3.11, pinned installs (numpy==1.26.4) may fail — see COLAB.md fallback.

In [ ]:
# 1) Get the repo. Option A (recommended): set REPO_URL to your GitHub repo and run.
#    Option B: upload the repo zip via Files panel, unzip it, and skip this cell.
REPO_URL = "https://github.com/Shanmukh-Villuri/loan-approval-predictor.git"  # prefilled — clone runs as-is on Colab
import os
if REPO_URL:
    !git clone $REPO_URL repo && mv repo/* repo/.* . 2>/dev/null; rm -rf repo
    print("cloned")
else:
    print("REPO_URL empty — assuming repo files already present (see COLAB.md option B).")
print(os.listdir("."))
assert os.path.exists("src/train.py"), "src/train.py not found — clone the repo or upload it first"

In [ ]:
# 2) Install pinned dependencies (matches .python-version 3.11.9).
%pip install -q -r requirements.txt
import sklearn, xgboost, imblearn, optuna
print("sklearn", sklearn.__version__, "| xgb", xgboost.__version__)

In [ ]:
# 3a) Data via Kaggle API (recommended — exact provenance). Upload kaggle.json when prompted.
#     Get kaggle.json from https://www.kaggle.com/settings (Account > API > Create New Token).
USE_KAGGLE_API = True
if USE_KAGGLE_API:
    from google.colab import files
    import shutil
    print("Upload your kaggle.json now:")
    up = files.upload()  # select kaggle.json
    os.makedirs("/root/.kaggle", exist_ok=True)
    shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
    os.makedirs("data", exist_ok=True)
    !kaggle competitions download -c home-credit-default-risk -f application_train.csv -p data/
    !unzip -o data/application_train.csv.zip -d data/ && rm data/application_train.csv.zip
    print("data:", os.listdir("data"))

In [ ]:
# 3b) Data via manual upload (fallback — no Kaggle token needed if you already have the CSV).
#     Run only if you skipped 3a. Upload application_train.csv (~166MB) when prompted.
USE_MANUAL_UPLOAD = False
if USE_MANUAL_UPLOAD:
    from google.colab import files
    os.makedirs("data", exist_ok=True)
    up = files.upload()  # select application_train.csv
    assert "application_train.csv" in up, "expected application_train.csv"
    shutil.move("application_train.csv", "data/application_train.csv")
    print("data:", os.listdir("data"))

In [ ]:
# 4) Verify dataset + post-encoding feature count (expect 307511x122, ~8% minority, ~260 features).
from src.data_loading import load_application_train, minority_rate
from src.preprocessing import add_engineered_features, infer_column_types, build_preprocessor, encoded_feature_count
df = load_application_train("data/application_train.csv")
print("raw_shape=", df.shape, "minority_rate=", round(float(minority_rate(df["TARGET"])), 4))
feat = add_engineered_features(df.drop(columns=["TARGET", "SK_ID_CURR"]))
num, cat = infer_column_types(feat)
pre = build_preprocessor(num, cat).fit(feat)
print("numeric=", len(num), "categorical=", len(cat), "encoded=", encoded_feature_count(pre))

In [ ]:
# 5) Full training: Optuna TPE (XGB 30 / RF 30 / SVC 20), SVD-50 benchmark, SMOTE-in-folds, stacking.
#    Takes HOURS on full data — keep the tab open. Quicker smoke test first if desired:
#    !python -m src.train --quick --out models_quick
!python -m src.train --data data/application_train.csv --out models

In [ ]:
# 6) Evaluate on held-out test + save plots.
!python -m src.evaluate --models-dir models --fig-dir reports/figures
!cat models/final_metrics.json

In [ ]:
# 7) Download results back to the laptop (commit models/*.json + reports/figures/*.png, update README table).
from google.colab import files
import glob
print(glob.glob("models/*.json"), glob.glob("reports/figures/*.png"))
files.download("models/final_metrics.json")
files.download("models/training_summary.json")
for p in sorted(glob.glob("reports/figures/*.png")):
    files.download(p)